In [2]:
import copy
import random

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

La partición se hace en dos pasos
 Primero se separa train con 80, después ese 20 restante se divide en partes iguales para obtener val y test con un 10 por ciento para cada una.

In [3]:
dataset = fetch_california_housing()
x = dataset.data
y = dataset.target
feature_names = dataset.feature_names

x_train, x_temp, y_train, y_temp = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

x_val, x_test, y_val, y_test = train_test_split(
    x_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

len(x_train), len(x_val), len(x_test), feature_names

(16512,
 2064,
 2064,
 ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude'])

In [4]:
x_train = torch.tensor(x_train, dtype=torch.float32)
x_val = torch.tensor(x_val, dtype=torch.float32)
x_test = torch.tensor(x_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_val = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

lat_idx = feature_names.index("Latitude")
lon_idx = feature_names.index("Longitude")
geo_idx = [lat_idx, lon_idx]
num_idx = [i for i in range(len(feature_names)) if i not in geo_idx]

x_train.shape, y_train.shape, num_idx, geo_idx

(torch.Size([16512, 8]), torch.Size([16512, 1]), [0, 1, 2, 3, 4, 5], [6, 7])

 nn.Module el módulo siguiente aprende sus estadísticas con train y las guarda como buffers y luego hace standard scaling sobre esas mismas columnas.

In [5]:
class HousingPreprocessor(nn.Module):
    def __init__(self, num_idx, total_features):
        super().__init__()
        self.num_idx = torch.tensor(num_idx, dtype=torch.long)
        self.total_features = total_features
        self.register_buffer("lower_bounds", torch.zeros(len(num_idx)))
        self.register_buffer("upper_bounds", torch.zeros(len(num_idx)))
        self.register_buffer("means", torch.zeros(len(num_idx)))
        self.register_buffer("stds", torch.ones(len(num_idx)))

    def fit(self, x):
        selected = x[:, self.num_idx]
        q1 = torch.quantile(selected, 0.25, dim=0)
        q3 = torch.quantile(selected, 0.75, dim=0)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        clipped = torch.clamp(selected, min=lower, max=upper)
        means = clipped.mean(dim=0)
        stds = clipped.std(dim=0, unbiased=False)
        stds = torch.where(stds < 1e-6, torch.ones_like(stds), stds)

        self.lower_bounds.copy_(lower)
        self.upper_bounds.copy_(upper)
        self.means.copy_(means)
        self.stds.copy_(stds)
        return self

    def forward(self, x):
        x = x.clone()
        selected = x[:, self.num_idx]
        selected = torch.maximum(selected, self.lower_bounds)
        selected = torch.minimum(selected, self.upper_bounds)
        selected = (selected - self.means) / self.stds
        x[:, self.num_idx] = selected
        return x

 Se usa una red totalmente conectada el preprocesamiento se coloca al inicio del forward para que cada predicción siga exactamente la misma ruta que siguieron los datos en entrenamiento.


In [6]:
class HousingRegressor(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout, num_idx):
        super().__init__()
        self.preprocessor = HousingPreprocessor(num_idx=num_idx, total_features=input_dim)

        layers = []
        current_dim = input_dim

        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(current_dim, hidden_dim))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            current_dim = hidden_dim

        layers.append(nn.Linear(current_dim, 1))
        self.network = nn.Sequential(*layers)

    def fit_preprocessor(self, x_train):
        self.preprocessor.fit(x_train)
        return self

    def forward(self, x):
        x = self.preprocessor(x)
        return self.network(x)

In [7]:
train_dataset = TensorDataset(x_train, y_train)
val_dataset = TensorDataset(x_val, y_val)
test_dataset = TensorDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

Entrenamiento y métricas Se usa MSELoss porque el objetivo es regresión y además permite optimizar directamente una métrica coherente con RMSE. Para elegir el mejor modelo se usa RMSE en validación no en prueba esa decisión separa el ajuste de hiperparámetros de la medición final y protege la estimación del desempeño real.

In [8]:
def regression_metrics(model, loader, device):
    model.eval()
    preds = []
    targets = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = model(xb)
            preds.append(pred)
            targets.append(yb)

    preds = torch.cat(preds, dim=0)
    targets = torch.cat(targets, dim=0)

    mse = torch.mean((preds - targets) ** 2)
    rmse = torch.sqrt(mse)
    mae = torch.mean(torch.abs(preds - targets))
    ss_res = torch.sum((targets - preds) ** 2)
    ss_tot = torch.sum((targets - torch.mean(targets)) ** 2)
    r2 = 1 - ss_res / ss_tot

    return {
        "MAE": mae.item(),
        "MSE": mse.item(),
        "RMSE": rmse.item(),
        "R2": r2.item()
    }


def train_model(model, train_loader, val_loader, epochs, lr, weight_decay, patience, device):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_rmse": []
    }

    best_state = copy.deepcopy(model.state_dict())
    best_val_rmse = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * xb.size(0)
            train_count += xb.size(0)

        train_loss = train_loss_sum / train_count
        val_metrics = regression_metrics(model, val_loader, device)
        val_loss = val_metrics["MSE"]
        val_rmse = val_metrics["RMSE"]

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_rmse"].append(val_rmse)

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)

    return model, history, best_val_rmse, best_epoch

Los tres modelos se diseñan para probar tres niveles distintos de capacidad. el primero busca una referencia compacta el segundo agrega profundidad y una regularización ligera para ver si mejora la generalización el tercero aumenta todavía más la capacidad, pero también sube el dropout y weight_decay para controlar el sobreajuste la comparación se hace con la misma semilla y el mismo conjunto de validación para que el cambio principal entre experimentos sean los hiperparámetros.

In [9]:
configs = [
    {
        "name": "modelo_1",
        "hidden_layers": [64, 32],
        "dropout": 0.0,
        "lr": 1e-3,
        "weight_decay": 0.0,
        "epochs": 200,
        "patience": 20
    },
    {
        "name": "modelo_2",
        "hidden_layers": [128, 64, 32],
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "epochs": 200,
        "patience": 20
    },
    {
        "name": "modelo_3",
        "hidden_layers": [256, 128, 64],
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "epochs": 200,
        "patience": 25
    }
]

results = []
trained_models = {}
histories = {}

for config in configs:
    torch.manual_seed(42)
    random.seed(42)

    model = HousingRegressor(
        input_dim=x_train.shape[1],
        hidden_layers=config["hidden_layers"],
        dropout=config["dropout"],
        num_idx=num_idx
    )

    model.fit_preprocessor(x_train)

    model, history, best_val_rmse, best_epoch = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=config["epochs"],
        lr=config["lr"],
        weight_decay=config["weight_decay"],
        patience=config["patience"],
        device=device
    )

    train_metrics = regression_metrics(model, train_loader, device)
    val_metrics = regression_metrics(model, val_loader, device)

    results.append({
        "name": config["name"],
        "hidden_layers": config["hidden_layers"],
        "dropout": config["dropout"],
        "lr": config["lr"],
        "weight_decay": config["weight_decay"],
        "best_epoch": best_epoch,
        "train_mae": train_metrics["MAE"],
        "train_rmse": train_metrics["RMSE"],
        "train_r2": train_metrics["R2"],
        "val_mae": val_metrics["MAE"],
        "val_rmse": val_metrics["RMSE"],
        "val_r2": val_metrics["R2"]
    })

    trained_models[config["name"]] = model
    histories[config["name"]] = history

results

[{'name': 'modelo_1',
  'hidden_layers': [64, 32],
  'dropout': 0.0,
  'lr': 0.001,
  'weight_decay': 0.0,
  'best_epoch': 27,
  'train_mae': 0.5186296701431274,
  'train_rmse': 0.6953516602516174,
  'train_r2': 0.6382991075515747,
  'val_mae': 0.5266978144645691,
  'val_rmse': 0.71835857629776,
  'val_r2': 0.6081010699272156},
 {'name': 'modelo_2',
  'hidden_layers': [128, 64, 32],
  'dropout': 0.1,
  'lr': 0.001,
  'weight_decay': 0.0001,
  'best_epoch': 156,
  'train_mae': 0.417638897895813,
  'train_rmse': 0.5890057682991028,
  'train_r2': 0.7404746413230896,
  'val_mae': 0.4226982891559601,
  'val_rmse': 0.606543242931366,
  'val_r2': 0.7206072807312012},
 {'name': 'modelo_3',
  'hidden_layers': [256, 128, 64],
  'dropout': 0.2,
  'lr': 0.0005,
  'weight_decay': 0.0005,
  'best_epoch': 61,
  'train_mae': 0.4613024890422821,
  'train_rmse': 0.652925431728363,
  'train_r2': 0.681090235710144,
  'val_mae': 0.4631468653678894,
  'val_rmse': 0.665896475315094,
  'val_r2': 0.66325211524